# NB4 — Medallion Pipeline (Bronze → Silver → Gold)

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `python /workspace/scripts/generate_data.py` (writes Bronze).

In [1]:
import sys
sys.path.append("/workspace/scripts")
from spark_session import get_spark, reset_path
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

spark = get_spark("nb4_medallion")

BRONZE = "s3a://bronze/llm_calls_raw"
SILVER = "s3a://silver/llm_calls"
GOLD   = "s3a://gold/llm_daily_metrics"

## Bronze — verify raw is loaded

In [2]:
bronze = spark.read.format("delta").load(BRONZE)
print("Bronze rows:", bronze.count())
bronze.printSchema()
bronze.show(2, truncate=80)

Bronze rows: 1000000
root
 |-- request_id: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- raw_json: string (nullable = true)



+------------------------------------+-------------------+--------------------------------------------------------------------------------+
|                          request_id|                 ts|                                                                        raw_json|
+------------------------------------+-------------------+--------------------------------------------------------------------------------+
|3333e9d3-7b74-4fa3-8d52-d966b36b9a1c|2026-04-04 22:26:44|{"model": "claude-sonnet-4-6", "user_id": "u_3461", "usage": {"input": 963, "...|
|9bcb67be-6e79-4b14-8f0c-e690cc4ffab3|2026-04-04 22:26:44|{"model": "claude-opus-4-7", "user_id": "u_1752", "usage": {"input": 2243, "o...|
+------------------------------------+-------------------+--------------------------------------------------------------------------------+
only showing top 2 rows



## Silver — parse, validate, dedup

Rules: drop rows with malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
parsed_schema = T.StructType([
    T.StructField("model", T.StringType()),
    T.StructField("user_id", T.StringType()),
    T.StructField("usage", T.StructType([
        T.StructField("input", T.IntegerType()),
        T.StructField("output", T.IntegerType()),
    ])),
    T.StructField("latency_ms", T.IntegerType()),
    T.StructField("status", T.StringType()),
])

silver_df = (
    bronze
    .withColumn("p", F.from_json("raw_json", parsed_schema))
    .where(F.col("p").isNotNull())
    .select(
        "request_id",
        "ts",
        F.col("p.model").alias("model"),
        F.col("p.user_id").alias("user_id"),
        F.col("p.usage.input").alias("prompt_tokens"),
        F.col("p.usage.output").alias("completion_tokens"),
        F.col("p.latency_ms").alias("latency_ms"),
        F.col("p.status").alias("status"),
        F.to_date("ts").alias("date"),
    )
    .dropDuplicates(["request_id"])
)

reset_path(spark, SILVER)
(silver_df.write.format("delta").mode("overwrite")
    .partitionBy("date")
    .save(SILVER))

bronze_n = bronze.count()
silver_n = spark.read.format("delta").load(SILVER).count()
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the updated generator (which seeds retries)?"
)

Silver rows: 949,981  (Bronze 1,000,000 → dedup dropped 50,019)


## Gold — aggregate to (date, model) metrics

In [4]:
silver = spark.read.format("delta").load(SILVER)

# Cost model — illustrative USD/M-token (NOT canonical)
COST = {
    "claude-haiku-4-5":   (0.80, 4.00),
    "claude-sonnet-4-6":  (3.00, 15.00),
    "claude-opus-4-7":    (15.00, 75.00),
}
cost_in  = F.create_map(*[x for k, v in COST.items() for x in (F.lit(k), F.lit(v[0]))])
cost_out = F.create_map(*[x for k, v in COST.items() for x in (F.lit(k), F.lit(v[1]))])

gold_df = (silver
    .groupBy("date", "model")
    .agg(
        F.percentile_approx("latency_ms", 0.5).alias("p50_latency_ms"),
        F.percentile_approx("latency_ms", 0.95).alias("p95_latency_ms"),
        F.sum("prompt_tokens").alias("total_prompt_tokens"),
        F.sum("completion_tokens").alias("total_completion_tokens"),
        (F.sum(F.when(F.col("status") != "ok", 1).otherwise(0))
            / F.count("*")).alias("error_rate"),
    )
    .withColumn(
        "cost_usd",
        (F.col("total_prompt_tokens")    * cost_in[F.col("model")]  / F.lit(1_000_000)) +
        (F.col("total_completion_tokens")* cost_out[F.col("model")] / F.lit(1_000_000))
    )
)

reset_path(spark, GOLD)
(gold_df.write.format("delta").mode("overwrite")
    .partitionBy("date")
    .save(GOLD))

# Z-ORDER by model for fast filter-by-model dashboards
spark.sql(f"OPTIMIZE delta.`{GOLD}` ZORDER BY (model)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,totalClusterPar

## Verify Gold

In [5]:
spark.read.format("delta").load(GOLD).orderBy("date", "model").show(20, truncate=False)

+----------+-----------------+--------------+--------------+-------------------+-----------------------+--------------------+------------------+
|date      |model            |p50_latency_ms|p95_latency_ms|total_prompt_tokens|total_completion_tokens|error_rate          |cost_usd          |
+----------+-----------------+--------------+--------------+-------------------+-----------------------+--------------------+------------------+
|2026-04-01|claude-haiku-4-5 |569           |1133          |82518522           |41097937               |0.050463338495194555|230.40656560000002|
|2026-04-01|claude-opus-4-7  |2997          |5948          |27649190           |13726288               |0.046470846120122755|1444.20945        |
|2026-04-01|claude-sonnet-4-6|1381          |2749          |164792985          |82090015               |0.050289067007082446|1725.7291799999998|
|2026-04-02|claude-haiku-4-5 |567           |1131          |82680930           |41265778               |0.04989834162114494 |231.2

## ✅ Deliverable check
- [ ] All three tables exist in MinIO (`bronze/`, `silver/`, `gold/`)
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold rows = (#dates × #models)
- [ ] Cost column populated (non-zero)

In [6]:
spark.stop()